In [53]:
import pandas as pd
import re



In [54]:
fk = pd.read_csv("fake.csv")
tr = pd.read_csv("true.csv")

In [55]:
fk.head()

,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017"
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017"


In [56]:
fk = fk[['title']]
tr = tr[['title']]
fk['label'] = 0
tr['label'] = 1

df = pd.concat([fk,tr])

In [57]:
df = df.sample(frac=1).reset_index(drop=True)

In [58]:
y = df.pop('label')
# df.drop("label",axis=1,inplace = True)
df.head()

,title
0,Swiss woman abducted in Sudan by criminal gang...
1,"U.N. brings Syria talks under one roof, not ye..."
2,"OBAMA’S GUN-RUNNING, Lying, Race-Baiting AG Is..."
3,SICKENING: Donald Trump Actually Hit On A 10-...
4,(VIDEO) AMAZING! TED CRUZ GETS CODE PINK TO SH...


In [59]:
type(df)

pandas.core.frame.DataFrame

In [60]:
from nltk.stem import PorterStemmer,WordNetLemmatizer
from nltk.corpus import stopwords
from nltk.tokenize import sent_tokenize,word_tokenize


In [61]:
df.isnull().sum()



title    0
dtype: int64

In [62]:
corpus = []
stem = PorterStemmer()
lem = WordNetLemmatizer()
for i in range(len(df)):
    words = str(df['title'][i]).lower()
    words = re.sub('[^a-zA-Z]',' ',words)
    words = words.split()
    ans = [stem.stem(word) for word in words if word not in stopwords.words('english')]
    ans = ' '.join(ans)
    corpus.append(ans)

In [63]:
# corpus = []
# h = "Boston Globe denounces Trump candidacy in 'front page' satire "
# words = h.lower()
# words = re.sub('[^a-zA-Z]',' ',words)   
# words = words.split()
# ans = [stem.stem(word) for word in words if word not in stopwords.words('english')]
# ans = ' '.join(ans)
# corpus.append(ans)

In [64]:
corpus

['swiss woman abduct sudan crimin gang ransom offici',
 'u n bring syria talk one roof yet one room',
 'obama gun run lie race bait ag back never guess whose job reportedli hope take video',
 'sicken donald trump actual hit year old girl video',
 'video amaz ted cruz get code pink shut listen iran nuke deal',
 'mike penc cancel anoth trump event campaign implod imag',
 'bannon role trump administr set critic firestorm',
 'virginia lawmak reach bipartisan deal gun issu',
 'iraqi pm offic say turkey agre deal baghdad oil export',
 'uk polic alert suspect packag london islington area',
 'watch reagan warn us draw battl line trump finish battl liber fascist video',
 'chuck schumer republican real problem trump behind close door video',
 'robert parri sort russia mess',
 'trump give slap press flynn guilti plea leav white hous video',
 'watch cheroke peopl express disgust lie elizabeth warren fake nativ american heritag get prestigi law professor job',
 'u homeland secretari kelli warn guat

In [65]:


from tensorflow.keras.preprocessing.text import one_hot
from tensorflow.keras.layers import Dense,Bidirectional,Embedding
from tensorflow.keras.models import Sequential


In [66]:
voc_size = 10000
onehot = [one_hot(words,voc_size) for words in corpus]


In [73]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
   

In [76]:
ma = 0
for i in onehot:
    ma = max(ma,len(i))

In [77]:
ma

35

In [78]:
max_seq = 35

In [79]:
padded = pad_sequences(onehot,maxlen=max_seq,padding = 'pre')

In [80]:
padded

array([[   0,    0,    0, ..., 5705, 5528, 9789],
       [   0,    0,    0, ..., 4203, 6224, 3937],
       [   0,    0,    0, ..., 8724, 6984, 5948],
       ...,
       [   0,    0,    0, ..., 2456, 8799, 4174],
       [   0,    0,    0, ..., 3026, 1992, 5948],
       [   0,    0,    0, ..., 9015, 8706, 4478]])

In [81]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(padded, y, test_size=0.25, random_state=42)

In [86]:
from tensorflow.keras.layers import LSTM
model = Sequential()
model.add(Embedding(voc_size,35,input_length = 35))
model.add(LSTM(100))
# model.add(Bidirectional(LSTM(100)))
model.add(Dense(1,activation = 'sigmoid'))
model.compile(loss = 'binary_crossentropy',optimizer = 'adam',metrics = ['accuracy'])

In [87]:
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [89]:
model.fit(x_train,y_train,epochs=5,batch_size=32)

Epoch 1/5


1053/1053 ━━━━━━━━━━━━━━━━━━━━ 45s 42ms/step - accuracy: 0.9675 - loss: 0.0869
Epoch 2/5
1053/1053 ━━━━━━━━━━━━━━━━━━━━ 39s 37ms/step - accuracy: 0.9800 - loss: 0.0564
Epoch 3/5
1053/1053 ━━━━━━━━━━━━━━━━━━━━ 46s 44ms/step - accuracy: 0.9866 - loss: 0.0392
Epoch 4/5
1053/1053 ━━━━━━━━━━━━━━━━━━━━ 43s 41ms/step - accuracy: 0.9899 - loss: 0.0287
Epoch 5/5
1053/1053 ━━━━━━━━━━━━━━━━━━━━ 37s 35ms/step - accuracy: 0.9930 - loss: 0.0215


In [90]:
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ (None, 35, 35)         │       350,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 100)            │        54,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           101 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,213,505 (4.63 MB)

 Trainable params: 404,501 (1.54 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 809,004 (3.09 MB)

In [97]:
from sklearn.metrics import accuracy_score,confusion_matrix
pred = model.predict(x_test)
pred_binary = (pred > 0.5).astype(int).reshape(-1)  # Convert probabilities to binary labels
acc = accuracy_score(y_test, pred_binary)
conf = confusion_matrix(y_test, pred_binary)

351/351 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step


In [98]:
acc

0.9472605790645879

In [99]:
conf

array([[5568,  258],
       [ 334, 5065]], dtype=int64)